# Final Model Evaluation: Swin-B + EVA-02 + CoAtNet with Metadata

This notebook implements the final evaluation pipeline for the proposed two-stage scalar temperature-scaling ensemble. It uses precomputed backbone feature arrays and integrates age and anatomical-region metadata.

## Experimental workflow

1. **Validation-stage model configuration:** train the classifier heads on the training split; select the CoAtNet logistic-regression parameter, member temperature parameters, fusion weights, and the second-stage ensemble temperature using validation data.
2. **Train+validation refit:** refit the classifier heads using the combined training and validation splits while keeping the selected hyperparameters and PCA component counts fixed.
3. **Final test prediction:** apply the frozen member temperatures, fusion weights, and ensemble temperature to the held-out test predictions.
4. **Publication reporting:** generate classification and calibration tables, stratified bootstrap confidence intervals, confusion matrices, ROC curves, and the reliability diagram.

The scientific workflow, model definitions, hyperparameters, calibration calculations, metrics, and reporting functions are retained from the supplied final-model code. Existing cell outputs are not included in this notebook.

## Data layout

Set the environment variable `SKIN_LESION_PROJECT_ROOT` to a project directory outside the Git repository. The code expects the following structure:

```text
<project-root>/
├── splits/
│   ├── train_split.csv
│   ├── val_split.csv
│   └── test_split.csv
├── features/
│   ├── swin/ft768/
│   ├── eva02/
│   └── coatnet/ft672/
├── metadata/
│   └── ISIC_2019_Training_Metadata.csv
└── outputs/   # generated automatically
```

Each feature directory must contain the expected train, validation, and test `.npy` arrays. Split CSV files must contain an image identifier column and either `label` or `class_name`; if both are present, their canonical mappings must agree.

**Feature-row alignment is required:** rows in every feature array must correspond exactly to the row order of the matching split CSV. This notebook loads feature arrays and labels separately and does not independently verify this alignment.

## Environment configuration

In Google Colab, mount Drive and set `SKIN_LESION_PROJECT_ROOT` before running the setup cell, for example:

```python
import os
os.environ["SKIN_LESION_PROJECT_ROOT"] = "/content/skin_lesion_project"
```

Install compatible versions of NumPy, pandas, scikit-learn, SciPy, Matplotlib, and PyTorch before running the notebook. Because the preserved source code passes `multi_class="auto"` to `LogisticRegression`, use scikit-learn 1.7.x or earlier unless that argument is updated and the results are revalidated. The training and reporting stages can require substantial memory and GPU resources.

## Output artifacts

The output directory contains `learned_parameters.json`, final predictions (`y_true.npy`, `y_pred.npy`, `y_proba.npy`), CSV/TXT/LaTeX tables, and publication figures in PNG, PDF, and SVG formats. These generated artifacts should remain outside the source-code repository unless selected for explicit publication.


## 1. Environment, Imports, and Configuration

This section configures deterministic execution, imports the required libraries, sets the configurable project paths, and defines the canonical eight-class mapping and fixed model hyperparameters. The `CUBLAS_WORKSPACE_CONFIG` setting remains before the PyTorch import as in the original code.

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
#  B12 Final Model Evaluation — Reproducible Research Notebook
#  - Architecture : Swin-B + EVA-02 + CoAtNet + Age (G3_NoMiss) + Region (R_full)
#  - Preprocessing: StandardScaler -> PCA (0.99 variance on Train, frozen k on Train+Val)
#  - Classifiers  : RobustMLP (Swin, EVA-02) + LogisticRegression (CoAtNet)
#  - Calibration  : Proposed Two-Stage Scalar Temperature Scaling
#                   Stage-1 (T_SWIN, T_EVA, T_COAT) -> NLL-Optimal Fusion Weights -> Stage-2 (T_ens)
#  - Uncertainty  : Class-Stratified Non-Parametric Bootstrap (B = 1000, 95% Percentile CI)
#  - Consistency  : Canonical 8-Class Mapping & Unified 15-Bin Calibration (ECE + Reliability)
#
#  FINAL PUBLICATION REPORTING
#    Table 1  : Overall classification performance + 95% Stratified Bootstrap CI
#    Table 2  : Per-class classification metrics (Precision, Sensitivity, Specificity, F1, Support)
#    Table 3  : Class-wise ROC-AUC + 95% Stratified Bootstrap CI (8 Classes + Macro & Micro AUC)
#    Table 4  : Calibration metrics (15-bin ECE, NLL, Brier) + 95% Stratified Bootstrap CI
#
#    Figure 1 : Confusion matrix (Fig1_ConfusionMatrix_RowNormalized + Supplementary Raw Counts)
#    Figure 2 : ROC curves (8 Classes One-vs-Rest + Macro & Micro Average)
#    Figure 3 : 15-Bin Reliability diagram of Proposed Two-Stage Scalar TS
#
#    Artifacts: learned_parameters.json (Full Reproducibility Manifest),
#               y_true.npy, y_pred.npy, y_proba.npy
# ============================================================

# ---- must be before torch
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import sys, platform, json, warnings, copy
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import sklearn

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, top_k_accuracy_score, log_loss,
    roc_auc_score, roc_curve, auc, confusion_matrix, precision_recall_fscore_support
)
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

# -------------------- Configurable Paths --------------------
# Set SKIN_LESION_PROJECT_ROOT to the directory containing the expected
# splits/, features/, and metadata/ subdirectories.
PROJECT_ROOT = Path(os.environ.get("SKIN_LESION_PROJECT_ROOT", "/content/skin_lesion_project"))

SPLIT_DIR = str(PROJECT_ROOT / "splits")
FEATURE_DIRS = {
    "SWIN": str(PROJECT_ROOT / "features" / "swin" / "ft768"),
    "EVA":  str(PROJECT_ROOT / "features" / "eva02"),
    "COAT": str(PROJECT_ROOT / "features" / "coatnet" / "ft672"),
}
META_CSV = str(PROJECT_ROOT / "metadata" / "ISIC_2019_Training_Metadata.csv")

SAVE_DIR = str(PROJECT_ROOT / "outputs" / "b12_final_single_model_outputs")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RANDOM_STATE = 42

USE_PCA = True
PCA_VARIANCE = 0.99
T_MIN, T_MAX, T_STEP = 0.70, 1.50, 0.02
STAGE2_T_STEP = 0.005
MLP_BATCH = 256

# Ensemble weight search
COAT_FLOOR = 0.05
W_STEP = 0.02

# Calibration bin count (shared 15 bins for ECE and reliability diagram)
N_CALIB_BINS = 15

# Number of bootstrap replicates and figure resolution
N_BOOTSTRAP = 1000
FIG_DPI = 600

# Canonical ISIC-2019 8-class mapping (required standard for all splits)
CANONICAL_CLASS_NAMES = ["AK", "BCC", "BKL", "DF", "MEL", "NV", "SCC", "VASC"]
CANONICAL_CLASS_MAP = {name: idx for idx, name in enumerate(CANONICAL_CLASS_NAMES)}
CLASS_NAME_ALIASES = {
    "AKIEC": "AK",
    "ACTINIC KERATOSIS": "AK",
    "BASAL CELL CARCINOMA": "BCC",
    "BENIGN KERATOSIS": "BKL",
    "DERMATOFIBROMA": "DF",
    "MELANOMA": "MEL",
    "MELANOCYTIC NEVUS": "NV",
    "NEVUS": "NV",
    "SQUAMOUS CELL CARCINOMA": "SCC",
    "VASCULAR LESION": "VASC",
    **{f"CLASS{i}": CANONICAL_CLASS_NAMES[i] for i in range(8)},
    **{f"CLASS_{i}": CANONICAL_CLASS_NAMES[i] for i in range(8)},
}

# Metadata configuration (G3_NoMiss + R_full)
CHAMPION_AGE_CFG = {
    "edges": [(0,10),(10,20),(20,30),(30,40),(40,50),(50,60),(60,70),(70,80),(80,np.inf)],
    "add_missing": False
}
CHAMPION_REGION_CFG = {"rare_merge": 0, "top_k": None, "custom_map": None}

# Preset hyperparameters and CoAtNet C-grid protocol
NAMED_PRESETS = {
    "SwinMLP": {"hidden":[2048], "dropout":0.1605757337, "lr":3.41409509895e-4, "weight_decay":1.488348195284e-4, "epochs":70, "log_every":10},
    "EvaMLP":  {"hidden":[2048], "dropout":0.3203382207442766, "lr":9.765455593461e-4, "weight_decay":8.212522451000109e-05, "epochs":70, "log_every":10},
}
HP_LOGREG = {"C":2.3402196063650043, "max_iter":587, "tol":3.6e-04, "solver":"lbfgs", "class_weight":None, "n_jobs":-1, "multi_class":"auto", "random_state":RANDOM_STATE}
COAT_C_GRID = [1.6, HP_LOGREG["C"], 3.2]


def set_seed(seed=RANDOM_STATE):
    import random
    import torch.backends.cudnn as cudnn
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    cudnn.benchmark = False
    cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=False)

set_seed(RANDOM_STATE)
os.makedirs(SAVE_DIR, exist_ok=True)
print("Device:", DEVICE)



## 2. Data and Metadata Loading

These helpers normalize image identifiers, validate the canonical class labels, load the predefined train/validation/test split files and precomputed feature arrays, and construct age and anatomical-region metadata features. The anatomical-region vocabulary is fitted using the training split.

In [ ]:
# -------------------- Data and Metadata Loading (Canonical Mapping) --------------------
def _glob_any(p: Path, names):
    for nm in names:
        f = p / nm
        if f.exists(): return str(f)
    return None

def _normalize_ids(arr_like):
    s = pd.Series(arr_like, dtype=str)
    s = s.str.replace(r"^.*[\\/]", "", regex=True)
    s = s.str.replace(r"\.(jpg|jpeg|png|bmp|tif|tiff|gif)$", "", regex=True)
    return s.values

def _canonicalize_class_token(val):
    tok = str(val).strip().upper()
    tok = CLASS_NAME_ALIASES.get(tok, tok)
    if tok not in CANONICAL_CLASS_MAP:
        raise ValueError(f"Unknown class label: '{val}'. Expected one of: {CANONICAL_CLASS_NAMES}")
    return CANONICAL_CLASS_MAP[tok]

def _read_split_csv(csv_path):
    df = pd.read_csv(csv_path)
    id_col = next((c for c in ["image_id","image","Image","filename","fname","img_id","name","file","img"] if c in df.columns), None)
    if id_col is None:
        raise ValueError(f"{csv_path}: no image_id/image column was found.")
    ids = _normalize_ids(df[id_col])

    y_from_label = None
    y_from_name  = None

    if "label" in df.columns:
        y_from_label = df["label"].astype(int).values
        if not np.all(np.isin(y_from_label, np.arange(len(CANONICAL_CLASS_NAMES)))):
            raise ValueError(f"{csv_path}: the 'label' column contains values outside 0..{len(CANONICAL_CLASS_NAMES)-1}.")

    if "class_name" in df.columns:
        y_from_name = np.array([_canonicalize_class_token(v) for v in df["class_name"].values], dtype=int)

    if y_from_label is not None and y_from_name is not None:
        if not np.array_equal(y_from_label, y_from_name):
            raise ValueError(f"{csv_path}: the 'label' and canonical 'class_name' columns do not match.")
        y = y_from_label
    elif y_from_label is not None:
        y = y_from_label
    elif y_from_name is not None:
        y = y_from_name
    else:
        raise ValueError(f"{csv_path}: neither a 'label' nor a 'class_name' column was found.")

    return ids, y, list(CANONICAL_CLASS_NAMES)

def load_global_splits(split_dir):
    paths = {"train": os.path.join(split_dir, "train_split.csv"),
             "val"  : os.path.join(split_dir, "val_split.csv"),
             "test" : os.path.join(split_dir, "test_split.csv")}
    out = {}
    for sp, pth in paths.items():
        ids, y, _ = _read_split_csv(pth)
        out[sp] = {"ids": ids, "y": y}
    return out, list(CANONICAL_CLASS_NAMES)

def _load_features(dir_path: str, split: str):
    p = Path(dir_path)
    X_path = _glob_any(p, [f"{split}_features.npy", f"features_{split}.npy", f"X_{split}.npy", f"{split}.npy"])
    if X_path is None: raise FileNotFoundError(f"{dir_path}: no {split} feature file was found.")
    return np.load(X_path)

AGE_ID_CANDS  = ["image_id","image","Image","filename","fname","name","file","img"]
AGE_COL_CANDS = ["age","Age","age_approx","patient_age","age_years","age_approximate"]
REGION_CANDS  = ["anatom_site_general","anatom_site","region","body_site","anatom_site_general_challenge"]

def _read_meta(meta_csv):
    m = pd.read_csv(meta_csv)
    id_col = next((c for c in AGE_ID_CANDS if c in m.columns), None)
    if id_col is None: raise ValueError("Metadata CSV: no image ID column was found.")
    m["_img"] = _normalize_ids(m[id_col].astype(str).values)

    age_col = next((c for c in AGE_COL_CANDS if c in m.columns), None)
    m["_age"] = np.nan if age_col is None else pd.to_numeric(m[age_col], errors="coerce").clip(lower=0)

    reg_col = next((c for c in REGION_CANDS if c in m.columns), None)
    if reg_col is None:
        m["_region_raw"] = "missing"
    else:
        reg = m[reg_col].astype(str).str.strip().str.lower().replace({"nan":"missing","none":"missing"})
        reg = reg.where(~reg.isin(["","nan","none","na"]), "missing")
        m["_region_raw"] = reg

    return m[["_img","_age","_region_raw"]].drop_duplicates("_img", keep="first")

def _age_onehot(a, edges):
    v = np.zeros(len(edges), dtype=np.float32)
    if a is None or np.isnan(a): return v
    for i,(lo,hi) in enumerate(edges):
        if (a>=lo) and (a<hi): v[i]=1.0; break
    return v

def make_region_vocab_and_mapper(ids_train, meta_df, scheme_cfg):
    mp = {r["_img"]: r["_region_raw"] for _, r in meta_df.iterrows()}
    regs_tr = pd.Series([mp.get(i, "missing") for i in ids_train], dtype=str).fillna("missing").str.lower().str.strip()

    cm = scheme_cfg.get("custom_map", None)
    if cm is not None:
        regs_tr = regs_tr.replace(cm)

    vc = regs_tr.value_counts()
    top_k = scheme_cfg.get("top_k", None)
    rare_merge = scheme_cfg.get("rare_merge", 0)
    if top_k is not None:
        keep = vc.index[:top_k].tolist()
    elif rare_merge and rare_merge>0:
        keep = vc[vc>=rare_merge].index.tolist()
    else:
        keep = vc.index.tolist()

    vocab = sorted(pd.Series(keep + ["other","missing"]).unique().tolist())
    if "missing" in vocab:
        vocab = [v for v in vocab if v!="missing"] + ["missing"]
    if "other" in vocab and (len(vocab)==0 or vocab[-1]!="missing"):
        vocab = [v for v in vocab if v not in ["other","missing"]] + ["other"] + (["missing"] if "missing" in vocab else [])

    def map_ids(ids_any):
        regs = pd.Series([mp.get(i, "missing") for i in ids_any], dtype=str).fillna("missing").str.lower().str.strip()
        if cm is not None:
            regs = regs.replace(cm)
        return regs.where(regs.isin(keep), "other").to_dict()

    return vocab, map_ids

def build_region_ohe(ids, id2map, vocab):
    idx = {v:i for i,v in enumerate(vocab)}
    M = np.zeros((len(ids), len(vocab)), dtype=np.float32)
    miss_idx = idx.get("missing", None)
    for r, img in enumerate(ids):
        key = id2map.get(img, "missing")
        j = idx.get(key, miss_idx)
        if j is not None: M[r, j] = 1.0
    return M

def build_age_block(ids, meta_df, edges, add_missing=True):
    age_mp = {r["_img"]: r["_age"] for _, r in meta_df.iterrows()}
    rows=[]
    for img in ids:
        a = age_mp.get(img, np.nan)
        one = _age_onehot(a, edges)
        if add_missing:
            miss = 0.0 if (a is not None and not np.isnan(a)) else 1.0
            one = np.concatenate([one, [miss]]).astype(np.float32)
        rows.append(one.astype(np.float32))
    return np.stack(rows, 0)

def augment(base_X, ids, meta_df, age_scheme_cfg, region_vocab, region_id2map):
    parts = [
        base_X,
        build_age_block(ids, meta_df, edges=age_scheme_cfg["edges"], add_missing=age_scheme_cfg["add_missing"]),
        build_region_ohe(ids, region_id2map, region_vocab)
    ]
    return np.hstack(parts).astype(np.float32)



## 3. Classifier Heads, Scaling, and PCA

This section defines the RobustMLP heads used for Swin-B and EVA-02, the CoAtNet logistic-regression helper, deterministic data loading, and the train-fitted scaling/PCA transformations.

In [ ]:
# -------------------- Models and Scaling/PCA --------------------
class RobustMLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(2048,), dropout=0.3):
        super().__init__()
        layers=[]; last=in_dim
        for h in hidden:
            layers += [nn.Linear(last,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            last=h
        layers += [nn.Linear(last, n_classes)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def _loader(X, y, bs, shuffle=True):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    g = torch.Generator(device="cpu"); g.manual_seed(RANDOM_STATE)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, generator=g, num_workers=0, drop_last=False)

def train_mlp(Xfit, yfit, n_classes, hp, device, tag, cosine=False):
    hp = copy.deepcopy(hp); hp["batch_size"] = MLP_BATCH
    model = RobustMLP(Xfit.shape[1], n_classes, tuple(hp["hidden"]), hp["dropout"]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, hp["epochs"])) if cosine else None
    crit = nn.CrossEntropyLoss(label_smoothing=0.05).to(device)
    loader = _loader(Xfit, yfit, hp["batch_size"], shuffle=True)
    model.train()
    for ep in range(1, hp["epochs"]+1):
        losses=[]
        for xb,yb in loader:
            xb=xb.to(device); yb=yb.to(device)
            opt.zero_grad(set_to_none=True)
            loss=crit(model(xb), yb); loss.backward(); opt.step()
            losses.append(loss.item())
        if sched is not None: sched.step()
        if ep==1 or ep%max(1,hp.get("log_every",10))==0 or ep==hp["epochs"]:
            with torch.no_grad():
                P = torch.softmax(model(torch.tensor(Xfit, dtype=torch.float32, device=device)), dim=1).cpu().numpy()
            acc = accuracy_score(yfit, P.argmax(1))
            f1  = f1_score(yfit, P.argmax(1), average="macro", zero_division=0)
            print(f"[{tag}] ep={ep:03d} | loss={np.mean(losses):.4f} | acc={acc:.4f} | f1={f1:.4f}")
    return model

def logits_proba(model, X, device):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.zeros(len(X), dtype=torch.long))
    loader = DataLoader(ds, batch_size=256, shuffle=False, num_workers=0, drop_last=False)
    model.eval(); outs=[]
    with torch.no_grad():
        for xb,_ in loader:
            outs.append(model(xb.to(device)).cpu())
    lg = torch.cat(outs,0).numpy()
    P  = torch.softmax(torch.tensor(lg), dim=1).numpy()
    return lg, P

def make_logreg(hp):
    hp = hp.copy()
    if hp.get("solver","lbfgs") == "lbfgs": hp.pop("n_jobs", None)
    return LogisticRegression(**hp)

def fit_transform_train_then_val(Xtr_aug, Xva_aug, use_pca=True, pca_var=0.99):
    sc = StandardScaler().fit(Xtr_aug)
    Ztr, Zva = sc.transform(Xtr_aug), sc.transform(Xva_aug)
    if not use_pca: return sc, None, Ztr, Zva, Ztr.shape[1]

    n_components = min(Xtr_aug.shape[0], Xtr_aug.shape[1])
    if isinstance(pca_var, float) and pca_var < 1.0:
        pca_n_arg = min(n_components, int(Xtr_aug.shape[0] * 0.99) - 1)
        solver = "randomized" if pca_n_arg < 0.9 * min(Xtr_aug.shape) else "full"
        pca_temp = PCA(n_components=max(2, pca_n_arg), svd_solver=solver, random_state=RANDOM_STATE).fit(Ztr)
        cumsum = np.cumsum(pca_temp.explained_variance_ratio_)
        k = np.argmax(cumsum >= pca_var) + 1
        pca = PCA(n_components=int(k), svd_solver="full", random_state=RANDOM_STATE).fit(Ztr)
    else:
        k = int(min(pca_var, n_components))
        pca = PCA(n_components=k, svd_solver="full", random_state=RANDOM_STATE).fit(Ztr)

    Ztr_p, Zva_p = pca.transform(Ztr), pca.transform(Zva)
    return sc, pca, Ztr_p, Zva_p, Ztr_p.shape[1]

def fit_transform_trainval_then_test_fixed(Xtr_aug, Xva_aug, Xte_aug, use_pca=True, pca_n=None, pca_var=0.99):
    Xtrval = np.vstack([Xtr_aug, Xva_aug])
    sc = StandardScaler().fit(Xtrval)
    Ztr, Zva, Zte = sc.transform(Xtr_aug), sc.transform(Xva_aug), sc.transform(Xte_aug)
    if not use_pca: return sc, None, Ztr, Zva, Zte

    Ztrval = np.vstack([Ztr, Zva])
    n_samples, n_features = Ztrval.shape
    k = int(min(pca_n, n_samples, n_features))
    pca = PCA(n_components=k, svd_solver="full", random_state=RANDOM_STATE).fit(Ztrval)
    return sc, pca, pca.transform(Ztr), pca.transform(Zva), pca.transform(Zte)



## 4. Calibration, Ensemble Fusion, and Metrics

This section implements temperature scaling, validation-NLL temperature selection, NLL-based ensemble-weight search, ECE calculation, and the classification/calibration metrics used by the final reporting pipeline.

In [ ]:
# -------------------- Calibration and Metric Functions --------------------
def temp_scale(P, T):
    P = np.clip(P, 1e-12, 1.0)
    T_arr = np.asarray(T, dtype=np.float64)
    if T_arr.ndim == 1:
        T_arr = T_arr[None, :]
    log_Q = np.log(P) / T_arr
    log_Q = log_Q - np.max(log_Q, axis=1, keepdims=True)
    Q = np.exp(log_Q)
    Q = Q / Q.sum(axis=1, keepdims=True)
    return Q

def pick_T_by_nll(y_true, P, n_class, tmin, tmax, tstep):
    grid = np.arange(tmin, tmax+1e-9, tstep)
    bestT, bestNLL = 1.0, 1e18
    for T in grid:
        Q = temp_scale(P, T)
        nll = log_loss(y_true, np.clip(Q, 1e-12, 1.0), labels=list(range(n_class)))
        if nll < bestNLL: bestNLL, bestT = nll, T
    return float(bestT), float(bestNLL)

def compute_ece(y_true, P, n_bins=N_CALIB_BINS):
    confidences = np.max(P, axis=1)
    predictions = np.argmax(P, axis=1)
    accuracies = (predictions == y_true).astype(float)
    bin_boundaries = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        bin_lower, bin_upper = bin_boundaries[i], bin_boundaries[i + 1]
        in_bin = (confidences >= bin_lower) & (confidences <= bin_upper) if i == 0 else (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = np.mean(in_bin)
        if prop_in_bin > 0:
            ece += np.abs(np.mean(confidences[in_bin]) - np.mean(accuracies[in_bin])) * prop_in_bin
    return float(ece)

def mix3(Pa, Pb, Pc, w_a, w_b):
    w_c = 1.0 - w_a - w_b
    return w_a*Pa + w_b*Pb + w_c*Pc

def weight_search_nll(P_val_dict, y_val, n_class):
    best = {"w_swin":0.54, "w_eva":0.32, "w_coat":0.14, "score":1e18}
    grid = np.arange(0.0, 1.0+1e-9, W_STEP)
    labels_list = list(range(n_class))
    for ws in grid:
        for we in grid:
            wc = 1.0 - ws - we
            if wc < COAT_FLOOR - 1e-12: continue
            w = np.array([max(0.0,ws), max(0.0,we), max(COAT_FLOOR,wc)], dtype=float); w/=w.sum()
            Pmix = mix3(P_val_dict["SWIN"], P_val_dict["EVA"], P_val_dict["COAT"], w[0], w[1])
            score = log_loss(y_val, np.clip(Pmix,1e-12,1.0), labels=labels_list)
            if score < best["score"] - 1e-12:
                best = {"w_swin":float(w[0]), "w_eva":float(w[1]), "w_coat":float(w[2]), "score":float(score)}
    return best

def eval_single_model_metrics(y_true, P, n_class):
    P_c = np.clip(P, 1e-12, 1.0)
    P_c = P_c / P_c.sum(axis=1, keepdims=True)
    y_pred = P_c.argmax(1)
    Y_ohe = np.eye(n_class)[y_true]
    labels_list = list(range(n_class))

    acc   = float(accuracy_score(y_true, y_pred))
    bacc  = float(balanced_accuracy_score(y_true, y_pred))
    mf1   = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    wf1   = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))
    top3  = float(top_k_accuracy_score(y_true, P_c, k=3, labels=labels_list))
    try:
        auc_macro = float(roc_auc_score(Y_ohe, P_c, average="macro", multi_class="ovr"))
    except Exception:
        auc_macro = np.nan

    ece   = compute_ece(y_true, P_c, n_bins=N_CALIB_BINS)
    nll   = float(log_loss(y_true, P_c, labels=labels_list))
    brier = float(np.mean(np.sum((P_c - Y_ohe) ** 2, axis=1)))

    return {
        "Accuracy": acc,
        "Balanced_Accuracy": bacc,
        "Macro_F1": mf1,
        "Weighted_F1": wf1,
        "Top3_Accuracy": top3,
        "Macro_ROC_AUC": auc_macro,
        "ECE": ece,
        "NLL": nll,
        "Brier": brier
    }



## 5. Bootstrap Confidence Intervals and Per-Class Reporting

These functions calculate class-stratified percentile-bootstrap confidence intervals, per-class precision/sensitivity/specificity/F1 metrics, and class-wise, macro, and micro ROC-AUC estimates.

In [ ]:
# -------------------- Class-Stratified 95% Bootstrap CIs and Per-Class Tables --------------------
def _precompute_class_indices(y_true, n_class):
    return [np.where(y_true == k)[0] for k in range(n_class)]

def _stratified_bootstrap_sample(class_indices, rng):
    """
    Generate stratified bootstrap indices by sampling, with replacement, the same
    number of observations from each class as its observed test-set support, thereby
    preserving the observed class counts in every replicate.
    """
    parts = [
        rng.choice(c_idx, size=len(c_idx), replace=True)
        for c_idx in class_indices if len(c_idx) > 0
    ]
    return np.concatenate(parts)

def bootstrap_final_model_ci(y_true, P, n_class, n_boot=N_BOOTSTRAP, seed=RANDOM_STATE):
    """
    Compute class-stratified 95% bootstrap confidence intervals for Table 1
    (overall performance) and Table 4 (calibration metrics).
    """
    rng = np.random.default_rng(seed)
    Pc = np.clip(P, 1e-12, 1.0); Pc /= Pc.sum(axis=1, keepdims=True)
    preds = Pc.argmax(1)
    Y_ohe = np.eye(n_class)[y_true]
    labels_list = list(range(n_class))
    class_indices = _precompute_class_indices(y_true, n_class)

    point_est = eval_single_model_metrics(y_true, Pc, n_class)
    metric_keys = [
        "Accuracy", "Balanced_Accuracy", "Macro_F1", "Weighted_F1",
        "Top3_Accuracy", "Macro_ROC_AUC", "ECE", "NLL", "Brier"
    ]
    boot_store = {mk: np.zeros(n_boot, dtype=np.float64) for mk in metric_keys}

    for b in range(n_boot):
        idx = _stratified_bootstrap_sample(class_indices, rng)
        yb = y_true[idx]
        Y_ohe_b = Y_ohe[idx]
        pb = Pc[idx]
        prb = preds[idx]

        boot_store["Accuracy"][b]          = np.mean(prb == yb)
        boot_store["Balanced_Accuracy"][b] = balanced_accuracy_score(yb, prb)
        boot_store["Macro_F1"][b]          = f1_score(yb, prb, average="macro", zero_division=0)
        boot_store["Weighted_F1"][b]       = f1_score(yb, prb, average="weighted", zero_division=0)
        boot_store["Top3_Accuracy"][b]     = top_k_accuracy_score(yb, pb, k=3, labels=labels_list)
        boot_store["Macro_ROC_AUC"][b]     = roc_auc_score(Y_ohe_b, pb, average="macro", multi_class="ovr")
        boot_store["ECE"][b]               = compute_ece(yb, pb, n_bins=N_CALIB_BINS)
        boot_store["NLL"][b]               = -np.mean(np.sum(Y_ohe_b * np.log(pb), axis=1))
        boot_store["Brier"][b]             = np.mean(np.sum((pb - Y_ohe_b) ** 2, axis=1))

    rows = []
    for mk in metric_keys:
        arr = boot_store[mk]
        arr_v = arr[~np.isnan(arr)]
        pt = float(point_est[mk])
        lo = float(np.percentile(arr_v, 2.5)) if len(arr_v) > 0 else np.nan
        hi = float(np.percentile(arr_v, 97.5)) if len(arr_v) > 0 else np.nan
        rows.append({
            "Metric": mk,
            "Estimate": pt,
            "CI95_Low": lo,
            "CI95_High": hi,
            "Formatted_95CI": f"{pt:.4f} [{lo:.4f}, {hi:.4f}]"
        })

    full_df = pd.DataFrame(rows)
    perf_keys = ["Accuracy", "Balanced_Accuracy", "Macro_F1", "Weighted_F1", "Top3_Accuracy", "Macro_ROC_AUC"]
    calib_keys = ["ECE", "NLL", "Brier"]

    table1_df = full_df[full_df["Metric"].isin(perf_keys)].reset_index(drop=True)
    table4_df = full_df[full_df["Metric"].isin(calib_keys)].reset_index(drop=True)
    return table1_df, table4_df

def build_per_class_report_with_specificity(y_true, y_pred, class_names):
    """
    Build Table 2 with per-class precision, sensitivity (recall), specificity,
    F1-score, and support.
    """
    n_class = len(class_names)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_class)))
    prec, rec, f1, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(n_class)), zero_division=0
    )

    total = cm.sum()
    specs = np.zeros(n_class, dtype=float)
    for k in range(n_class):
        tp = cm[k, k]
        fp = cm[:, k].sum() - tp
        fn = cm[k, :].sum() - tp
        tn = total - (tp + fp + fn)
        specs[k] = float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0

    rows = []
    for k, cname in enumerate(class_names):
        rows.append({
            "Class": cname,
            "Precision": float(prec[k]),
            "Sensitivity": float(rec[k]),
            "Specificity": float(specs[k]),
            "F1_Score": float(f1[k]),
            "Support": int(sup[k])
        })

    weights = sup / np.maximum(sup.sum(), 1)
    rows.append({
        "Class": "Macro_Avg",
        "Precision": float(np.mean(prec)),
        "Sensitivity": float(np.mean(rec)),
        "Specificity": float(np.mean(specs)),
        "F1_Score": float(np.mean(f1)),
        "Support": int(sup.sum())
    })
    rows.append({
        "Class": "Weighted_Avg",
        "Precision": float(np.sum(prec * weights)),
        "Sensitivity": float(np.sum(rec * weights)),
        "Specificity": float(np.sum(specs * weights)),
        "F1_Score": float(np.sum(f1 * weights)),
        "Support": int(sup.sum())
    })
    return pd.DataFrame(rows)

def bootstrap_classwise_roc_auc(y_true, P, class_names, n_boot=N_BOOTSTRAP, seed=RANDOM_STATE):
    """
    Compute per-class one-vs-rest ROC-AUC, macro-AUC, micro-AUC, and
    class-stratified 95% bootstrap confidence intervals for Table 3.
    """
    rng = np.random.default_rng(seed)
    n_class = len(class_names)
    N = len(y_true)
    Pc = np.clip(P, 1e-12, 1.0); Pc /= Pc.sum(axis=1, keepdims=True)
    Y_ohe = np.eye(n_class)[y_true]
    class_indices = _precompute_class_indices(y_true, n_class)

    point_aucs, supports = {}, {}
    for k, cname in enumerate(class_names):
        supports[cname] = int(np.sum(Y_ohe[:, k] == 1))
        point_aucs[cname] = float(roc_auc_score(Y_ohe[:, k], Pc[:, k]))
    point_aucs["Macro_AUC"] = float(roc_auc_score(Y_ohe, Pc, average="macro", multi_class="ovr"))
    point_aucs["Micro_AUC"] = float(roc_auc_score(Y_ohe, Pc, average="micro", multi_class="ovr"))
    supports["Macro_AUC"] = N
    supports["Micro_AUC"] = N

    all_keys = list(class_names) + ["Macro_AUC", "Micro_AUC"]
    boot_aucs = {k: np.full(n_boot, np.nan, dtype=np.float64) for k in all_keys}

    for b in range(n_boot):
        idx = _stratified_bootstrap_sample(class_indices, rng)
        Yb = Y_ohe[idx]
        pb = Pc[idx]
        valid_k = []
        for k, cname in enumerate(class_names):
            ak = roc_auc_score(Yb[:, k], pb[:, k])
            boot_aucs[cname][b] = ak
            valid_k.append(ak)
        boot_aucs["Macro_AUC"][b] = float(np.mean(valid_k))
        boot_aucs["Micro_AUC"][b] = float(roc_auc_score(Yb, pb, average="micro", multi_class="ovr"))

    rows = []
    for k in all_keys:
        arr = boot_aucs[k]
        arr_v = arr[~np.isnan(arr)]
        pt = point_aucs[k]
        lo = float(np.percentile(arr_v, 2.5)) if len(arr_v) > 0 else np.nan
        hi = float(np.percentile(arr_v, 97.5)) if len(arr_v) > 0 else np.nan
        rows.append({
            "Class": k,
            "Support": supports[k],
            "ROC_AUC": pt,
            "CI95_Low": lo,
            "CI95_High": hi,
            "Formatted_95CI": f"{pt:.4f} [{lo:.4f}, {hi:.4f}]"
        })
    return pd.DataFrame(rows)



## 6. Publication Figures and Table Export

This section generates the confusion matrices, one-vs-rest ROC curves, reliability diagram, and CSV/TXT/LaTeX versions of the final publication tables.

In [ ]:
# -------------------- Publication Figures (600-DPI PNG / PDF / SVG) and LaTeX Export --------------------
def _save_pub_fig(fig, fig_dir, base_name):
    os.makedirs(fig_dir, exist_ok=True)
    for ext in ["png", "pdf", "svg"]:
        pth = os.path.join(fig_dir, f"{base_name}.{ext}")
        fig.savefig(pth, dpi=FIG_DPI if ext == "png" else None, bbox_inches="tight")
    plt.close(fig)

def plot_figure1_confusion_matrices(fig_dir, y_true, y_pred, class_names):
    cm_raw = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    with np.errstate(divide="ignore", invalid="ignore"):
        cm_norm = cm_raw.astype(float) / cm_raw.sum(axis=1, keepdims=True)
        cm_norm = np.nan_to_num(cm_norm, nan=0.0)

    # Figure 1 (Main Manuscript): Row-Normalized Confusion Matrix (%)
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    im = ax.imshow(cm_norm * 100.0, interpolation="nearest", cmap="Blues", vmin=0.0, vmax=100.0)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel("Row-Normalized Sensitivity (%)", rotation=270, labelpad=15, fontsize=10)

    ax.set(
        xticks=np.arange(len(class_names)), yticks=np.arange(len(class_names)),
        xticklabels=class_names, yticklabels=class_names,
        ylabel="True Diagnosis", xlabel="Predicted Diagnosis",
        title="Row-Normalized Confusion Matrix (%)"
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor", fontsize=10)
    plt.setp(ax.get_yticklabels(), fontsize=10)

    for i in range(cm_norm.shape[0]):
        for j in range(cm_norm.shape[1]):
            val_pct = cm_norm[i, j] * 100.0
            ax.text(j, i, f"{val_pct:.1f}%", ha="center", va="center",
                    color="white" if val_pct > 50.0 else "black", fontsize=8.5,
                    fontweight="bold" if i == j else "normal")
    fig.tight_layout()
    _save_pub_fig(fig, fig_dir, "Fig1_ConfusionMatrix_RowNormalized")

    # Supplementary: Raw Count Confusion Matrix
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    im = ax.imshow(cm_raw, interpolation="nearest", cmap="Blues")
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel("Number of Test Images", rotation=270, labelpad=15, fontsize=10)

    ax.set(
        xticks=np.arange(len(class_names)), yticks=np.arange(len(class_names)),
        xticklabels=class_names, yticklabels=class_names,
        ylabel="True Diagnosis", xlabel="Predicted Diagnosis",
        title="Confusion Matrix (Raw Sample Counts)"
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor", fontsize=10)
    plt.setp(ax.get_yticklabels(), fontsize=10)

    thresh_raw = cm_raw.max() / 2.0
    for i in range(cm_raw.shape[0]):
        for j in range(cm_raw.shape[1]):
            ax.text(j, i, f"{cm_raw[i, j]:d}", ha="center", va="center",
                    color="white" if cm_raw[i, j] > thresh_raw else "black", fontsize=9,
                    fontweight="bold" if i == j else "normal")
    fig.tight_layout()
    _save_pub_fig(fig, fig_dir, "Supp_Fig1_ConfusionMatrix_RawCounts")

    pd.DataFrame(cm_norm, index=[f"true_{c}" for c in class_names], columns=[f"pred_{c}" for c in class_names]).to_csv(
        os.path.join(fig_dir, "CM_RowNormalized.csv")
    )
    pd.DataFrame(cm_raw, index=[f"true_{c}" for c in class_names], columns=[f"pred_{c}" for c in class_names]).to_csv(
        os.path.join(fig_dir, "CM_RawCounts.csv")
    )

def plot_figure2_roc_curves(fig_dir, y_true, P, class_names):
    n_class = len(class_names)
    Pc = np.clip(P, 1e-12, 1.0); Pc /= Pc.sum(axis=1, keepdims=True)
    Y_bin = label_binarize(y_true, classes=list(range(n_class)))

    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(n_class):
        fpr[i], tpr[i], _ = roc_curve(Y_bin[:, i], Pc[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    fpr["micro"], tpr["micro"], _ = roc_curve(Y_bin.ravel(), Pc.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_class)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_class):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_class
    fpr["macro"], tpr["macro"] = all_fpr, mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

    fig, ax = plt.subplots(figsize=(8.0, 6.8))
    ax.plot(fpr["macro"], tpr["macro"],
            label=f"Macro-average ROC (AUC = {roc_auc['macro']:.4f})",
            color="navy", linestyle="--", linewidth=2.5)
    ax.plot(fpr["micro"], tpr["micro"],
            label=f"Micro-average ROC (AUC = {roc_auc['micro']:.4f})",
            color="deeppink", linestyle=":", linewidth=2.5)

    cmap = plt.get_cmap("tab10")
    for i, cname in enumerate(class_names):
        ax.plot(fpr[i], tpr[i], color=cmap(i % 10), lw=1.6, alpha=0.9,
                label=f"{cname} (AUC = {roc_auc[i]:.4f})")

    ax.plot([0, 1], [0, 1], "k--", lw=1.0, alpha=0.6, label="Chance (AUC = 0.5000)")
    ax.set_xlim([-0.01, 1.0])
    ax.set_ylim([0.0, 1.02])
    ax.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
    ax.set_ylabel("True Positive Rate (Sensitivity)", fontsize=11)
    ax.set_title("One-vs-Rest Multi-Class ROC Curves on ISIC-2019 Test Set", fontsize=12)
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend(loc="lower right", fontsize=8.8, frameon=True)
    fig.tight_layout()
    _save_pub_fig(fig, fig_dir, "Fig2_ROC_Curves_OvR")

def plot_figure3_reliability_diagram(fig_dir, y_true, P, n_class, n_bins=N_CALIB_BINS):
    Pc = np.clip(P, 1e-12, 1.0); Pc /= Pc.sum(axis=1, keepdims=True)
    mt = eval_single_model_metrics(y_true, Pc, n_class)

    conf = np.max(Pc, axis=1)
    pred = np.argmax(Pc, axis=1)
    acc  = (pred == y_true).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)

    bin_confs, bin_accs, bin_counts = [], [], []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (conf >= lo) & (conf <= hi) if i == 0 else (conf > lo) & (conf <= hi)
        if np.sum(mask) > 0:
            bin_confs.append(float(np.mean(conf[mask])))
            bin_accs.append(float(np.mean(acc[mask])))
            bin_counts.append(int(np.sum(mask)))

    bin_confs = np.array(bin_confs)
    bin_accs  = np.array(bin_accs)

    fig, ax = plt.subplots(figsize=(6.8, 6.0))
    ax.plot([0, 1], [0, 1], "k--", lw=1.4, alpha=0.75, label="Perfect Calibration")
    ax.plot(bin_confs, bin_accs, marker="D", linestyle="-", color="#2ca02c",
            lw=2.4, markersize=6.0, label=f"Proposed Two-Stage Scalar TS ({n_bins}-bin)")

    info_text = (
        f"ECE ({n_bins}-bin) = {mt['ECE']:.4f}\n"
        f"NLL          = {mt['NLL']:.4f}\n"
        f"Brier Score  = {mt['Brier']:.4f}"
    )
    ax.text(
        0.05, 0.88, info_text, transform=ax.transAxes, fontsize=10,
        verticalalignment="top", family="monospace",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#f8f9fa", edgecolor="#2ca02c", lw=1.2)
    )

    ax.set_xlim([0.20, 1.02])
    ax.set_ylim([0.20, 1.02])
    ax.set_xlabel("Mean Predicted Confidence", fontsize=11)
    ax.set_ylabel("Empirical Accuracy", fontsize=11)
    ax.set_title(f"Reliability Diagram of Proposed Two-Stage Scalar TS ({n_bins} Bins)", fontsize=12)
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend(loc="lower right", fontsize=9.5, frameon=True)
    fig.tight_layout()
    _save_pub_fig(fig, fig_dir, "Fig3_Reliability_Diagram")

def export_publication_tables(out_dir, latex_dir, table1_df, table2_df, table3_df, table4_df):
    os.makedirs(latex_dir, exist_ok=True)

    # Table 1: Overall performance + 95% Stratified Bootstrap CI
    table1_df.to_csv(os.path.join(out_dir, "Table1_Overall_Performance_95CI.csv"), index=False)
    with open(os.path.join(latex_dir, "Table1_Overall_Performance_95CI.tex"), "w", encoding="utf-8") as f:
        f.write(table1_df.to_latex(
            index=False, float_format="%.4f",
            caption="Overall Classification Performance of the Proposed Model on the ISIC-2019 Test Set (with 95% Class-Stratified Bootstrap CIs)",
            label="tab:overall_performance"
        ))

    # Table 2: Per-class classification metrics (.csv, .txt, .tex)
    table2_df.to_csv(os.path.join(out_dir, "Table2_Classification_Report.csv"), index=False)
    with open(os.path.join(out_dir, "Table2_Classification_Report.txt"), "w", encoding="utf-8") as f:
        f.write(table2_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    with open(os.path.join(latex_dir, "Table2_Classification_Report.tex"), "w", encoding="utf-8") as f:
        f.write(table2_df.to_latex(
            index=False, float_format="%.4f",
            caption="Per-Class Classification Metrics (Precision, Sensitivity, Specificity, F1-Score, Support) on the ISIC-2019 Test Set",
            label="tab:classification_report"
        ))

    # Table 3: Class-wise ROC-AUC + 95% Stratified Bootstrap CI
    table3_df.to_csv(os.path.join(out_dir, "Table3_Classwise_ROCAUC_95CI.csv"), index=False)
    with open(os.path.join(latex_dir, "Table3_Classwise_ROCAUC_95CI.tex"), "w", encoding="utf-8") as f:
        f.write(table3_df.to_latex(
            index=False, float_format="%.4f",
            caption="Class-Wise One-vs-Rest, Macro, and Micro ROC-AUC Scores with 95% Class-Stratified Bootstrap CIs",
            label="tab:classwise_roc_auc"
        ))

    # Table 4: Calibration metrics + 95% Stratified Bootstrap CI
    table4_df.to_csv(os.path.join(out_dir, "Table4_Calibration_Metrics_95CI.csv"), index=False)
    with open(os.path.join(latex_dir, "Table4_Calibration_Metrics_95CI.tex"), "w", encoding="utf-8") as f:
        f.write(table4_df.to_latex(
            index=False, float_format="%.4f",
            caption=f"Calibration Performance ({N_CALIB_BINS}-Bin ECE, NLL, Brier Score) of the Proposed Model with 95% Class-Stratified Bootstrap CIs",
            label="tab:calibration_metrics"
        ))



## 7. Main Pipeline: Validation, Refit, Final Test, and Reporting

The final section runs the four-stage workflow. Model and calibration choices are made using the validation split; the selected settings are then frozen for train+validation refitting and final test evaluation. The code in this section is retained from the supplied final-model implementation.

In [ ]:
# -------------------- Main Pipeline (Four Stages) --------------------
def main():
    set_seed(RANDOM_STATE)
    splits, class_names = load_global_splits(SPLIT_DIR)
    n_class = len(class_names)

    data = {}
    for feat, path in FEATURE_DIRS.items():
        data[feat] = {
            "Xtr": _load_features(path, "train"),
            "Xva": _load_features(path, "val"),
            "Xte": _load_features(path, "test"),
            "ytr": splits["train"]["y"],
            "yva": splits["val"]["y"],
            "yte": splits["test"]["y"],
        }

    meta_df = _read_meta(META_CSV)

    # Region vocabulary and mappers are fitted using the training split only
    region_vocab, map_ids = make_region_vocab_and_mapper(splits["train"]["ids"], meta_df, CHAMPION_REGION_CFG)
    idmap_tr = map_ids(splits["train"]["ids"])
    idmap_va = map_ids(splits["val"]["ids"])
    idmap_te = map_ids(splits["test"]["ids"])

    ids_tr, ids_va, ids_te = splits["train"]["ids"], splits["val"]["ids"], splits["test"]["ids"]
    y_tr  = splits["train"]["y"]
    y_val = splits["val"]["y"]
    y_te  = splits["test"]["y"]

    # =========================================================================
    # STAGE 1 — VALIDATION (T_SWIN, T_EVA, T_COAT, fusion weights, T_ens)
    # =========================================================================
    print("\n" + "="*76)
    print("STAGE 1: TRAINING ON TRAIN SPLIT & LEARNING CALIBRATION/FUSION ON VALIDATION")
    print("="*76)

    P_val, T_sel, aux = {}, {}, {}
    raw_feature_dims, aug_feature_dims = {}, {}

    for memb in ["SWIN", "EVA", "COAT"]:
        Xtr_aug = augment(data[memb]["Xtr"], ids_tr, meta_df, CHAMPION_AGE_CFG, region_vocab, idmap_tr)
        Xva_aug = augment(data[memb]["Xva"], ids_va, meta_df, CHAMPION_AGE_CFG, region_vocab, idmap_va)
        raw_feature_dims[memb] = int(data[memb]["Xtr"].shape[1])
        aug_feature_dims[memb] = int(Xtr_aug.shape[1])

        sc, pca, Ztr, Zva, k = fit_transform_train_then_val(Xtr_aug, Xva_aug, USE_PCA, PCA_VARIANCE)

        if memb in ["SWIN", "EVA"]:
            hp = copy.deepcopy(NAMED_PRESETS["SwinMLP" if memb == "SWIN" else "EvaMLP"])
            mdl = train_mlp(Ztr, data[memb]["ytr"], n_class, hp, DEVICE, f"{memb} TRAIN->VAL")
            _, P_raw = logits_proba(mdl, Zva, DEVICE)
            T_i, _ = pick_T_by_nll(y_val, P_raw, n_class, T_MIN, T_MAX, T_STEP)
            P_val[memb] = temp_scale(P_raw, T_i)
            T_sel[memb] = float(T_i)
            aux[memb]   = {"hp": hp, "pca_n": int(k)}
        else:
            # COAT C-Grid Search on Validation NLL
            bestC, bestNLL, bestP, bestT = None, 1e18, None, None
            for C in COAT_C_GRID:
                hpC = copy.deepcopy(HP_LOGREG); hpC.update({"class_weight": "balanced", "C": float(C)})
                lg = make_logreg(hpC).fit(Ztr, data["COAT"]["ytr"])
                P_raw = lg.predict_proba(Zva)
                Tc, _ = pick_T_by_nll(y_val, P_raw, n_class, T_MIN, T_MAX, T_STEP)
                P_cal = temp_scale(P_raw, Tc)
                nll = log_loss(y_val, np.clip(P_cal, 1e-12, 1.0), labels=list(range(n_class)))
                if nll < bestNLL:
                    bestNLL, bestC, bestP, bestT = nll, float(C), P_cal, float(Tc)

            hpC_final = copy.deepcopy(HP_LOGREG); hpC_final.update({"class_weight": "balanced", "C": bestC})
            P_val["COAT"] = bestP.copy()
            T_sel["COAT"] = bestT
            aux["COAT"]   = {"hp": hpC_final, "pca_n": int(k)}

    # Fusion weights and Stage-2 T_ens based on Stage-1 calibrated probabilities
    w_stage1 = weight_search_nll(P_val, y_val, n_class)
    P_ens_val_stage1 = mix3(P_val["SWIN"], P_val["EVA"], P_val["COAT"], w_stage1["w_swin"], w_stage1["w_eva"])
    T_ens, val_final_nll = pick_T_by_nll(y_val, P_ens_val_stage1, n_class, T_MIN, T_MAX, STAGE2_T_STEP)

    # Full reproducibility manifest (learned_parameters.json)
    age_edges_serializable = [[float(lo), "inf" if np.isinf(hi) else float(hi)] for (lo, hi) in CHAMPION_AGE_CFG["edges"]]
    learned_params = {
        "model_name": "Proposed_TwoStage_Scalar_TS (A4_CHAMPION_G3_R_full)",
        "random_state": int(RANDOM_STATE),
        "canonical_class_names": class_names,
        "canonical_class_map": CANONICAL_CLASS_MAP,
        "split_sample_counts": {
            "train": int(len(y_tr)),
            "val":   int(len(y_val)),
            "test":  int(len(y_te)),
            "total": int(len(y_tr) + len(y_val) + len(y_te))
        },
        "split_class_counts": {
            sp_name: {class_names[k]: int(np.sum(sp_y == k)) for k in range(n_class)}
            for sp_name, sp_y in [("train", y_tr), ("val", y_val), ("test", y_te)]
        },
        "feature_dimensions": {
            "raw_backbone_dims": raw_feature_dims,
            "augmented_dims_with_meta": aug_feature_dims,
            "pca_retained_dims": {m: int(aux[m]["pca_n"]) for m in ["SWIN", "EVA", "COAT"]},
            "use_pca": bool(USE_PCA),
            "pca_variance_threshold": float(PCA_VARIANCE)
        },
        "metadata_configuration": {
            "age_edges": age_edges_serializable,
            "age_add_missing_indicator": bool(CHAMPION_AGE_CFG["add_missing"]),
            "region_scheme": "R_full",
            "region_vocabulary": region_vocab,
            "region_vocab_size": int(len(region_vocab))
        },
        "classifier_hyperparameters": {
            "SWIN_MLP": {**aux["SWIN"]["hp"], "batch_size": int(MLP_BATCH), "label_smoothing": 0.05},
            "EVA_MLP":  {**aux["EVA"]["hp"],  "batch_size": int(MLP_BATCH), "label_smoothing": 0.05},
            "COAT_LogReg": aux["COAT"]["hp"],
            "COAT_C_Grid_Evaluated": [float(c) for c in COAT_C_GRID],
            "COAT_Selected_C": float(aux["COAT"]["hp"]["C"])
        },
        "calibration_and_fusion": {
            "temperature_grid_stage1": {"min": float(T_MIN), "max": float(T_MAX), "step": float(T_STEP)},
            "temperature_grid_stage2": {"min": float(T_MIN), "max": float(T_MAX), "step": float(STAGE2_T_STEP)},
            "weight_search_grid": {"step": float(W_STEP), "coat_floor": float(COAT_FLOOR)},
            "stage1_temperatures": {m: float(T_sel[m]) for m in ["SWIN", "EVA", "COAT"]},
            "fusion_weights": {
                "w_swin": float(w_stage1["w_swin"]),
                "w_eva":  float(w_stage1["w_eva"]),
                "w_coat": float(w_stage1["w_coat"]),
                "val_stage1_nll": float(w_stage1["score"])
            },
            "stage2_temperature_T_ens": float(T_ens),
            "val_final_nll": float(val_final_nll)
        },
        "evaluation_protocol": {
            "ece_and_reliability_bins": int(N_CALIB_BINS),
            "bootstrap_replicates": int(N_BOOTSTRAP),
            "bootstrap_sampling": "Class-Stratified Non-Parametric Percentile Bootstrap (95% CI)"
        },
        "software_versions": {
            "python": sys.version.split()[0],
            "platform": platform.platform(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "scikit_learn": sklearn.__version__,
            "torch": torch.__version__,
            "cuda": torch.version.cuda if torch.cuda.is_available() else "CPU"
        }
    }
    with open(os.path.join(SAVE_DIR, "learned_parameters.json"), "w", encoding="utf-8") as f:
        json.dump(learned_params, f, indent=2)

    print("\n[LEARNED PARAMETERS & REPRODUCIBILITY MANIFEST SAVED]")
    print(f"  PCA Retained Dims    : {learned_params['feature_dimensions']['pca_retained_dims']}")
    print(f"  CoAtNet Selected C   : {learned_params['classifier_hyperparameters']['COAT_Selected_C']}")
    print(f"  Stage-1 Temperatures : {learned_params['calibration_and_fusion']['stage1_temperatures']}")
    print(f"  Fusion Weights       : {learned_params['calibration_and_fusion']['fusion_weights']}")
    print(f"  Stage-2 T_ens        : {T_ens:.4f} (Val Final NLL = {val_final_nll:.5f})")

    # =========================================================================
    # STAGE 2 — TRAIN+VALIDATION REFIT (frozen PCA k and hyperparameters)
    # =========================================================================
    print("\n" + "="*76)
    print("STAGE 2: REFITTING BACKBONE HEADS ON TRAIN+VAL SPLIT")
    print("="*76)

    P_test_stage1 = {}
    for memb in ["SWIN", "EVA", "COAT"]:
        Xtr_aug = augment(data[memb]["Xtr"], ids_tr, meta_df, CHAMPION_AGE_CFG, region_vocab, idmap_tr)
        Xva_aug = augment(data[memb]["Xva"], ids_va, meta_df, CHAMPION_AGE_CFG, region_vocab, idmap_va)
        Xte_aug = augment(data[memb]["Xte"], ids_te, meta_df, CHAMPION_AGE_CFG, region_vocab, idmap_te)

        sc, pca, Ztr, Zva, Zte = fit_transform_trainval_then_test_fixed(
            Xtr_aug, Xva_aug, Xte_aug, USE_PCA, aux[memb]["pca_n"], PCA_VARIANCE
        )
        Ztrval = np.vstack([Ztr, Zva])
        ytrval = np.concatenate([data[memb]["ytr"], data[memb]["yva"]])

        if memb in ["SWIN", "EVA"]:
            mdl = train_mlp(Ztrval, ytrval, n_class, aux[memb]["hp"], DEVICE, f"{memb} TRAIN+VAL->TEST")
            _, P_raw = logits_proba(mdl, Zte, DEVICE)
        else:
            lg = make_logreg(aux["COAT"]["hp"]).fit(Ztrval, ytrval)
            P_raw = lg.predict_proba(Zte)

        # Apply frozen Stage-1 temperature scaling
        P_test_stage1[memb] = temp_scale(P_raw, T_sel[memb])

    # =========================================================================
    # STAGE 3 — FINAL TEST PREDICTION (frozen fusion weights and Stage-2 T_ens)
    # =========================================================================
    P_ens_test_stage1 = mix3(
        P_test_stage1["SWIN"], P_test_stage1["EVA"], P_test_stage1["COAT"],
        w_stage1["w_swin"], w_stage1["w_eva"]
    )
    P_final      = temp_scale(P_ens_test_stage1, T_ens)
    y_pred_final = P_final.argmax(1)

    # Save the core NPY artifacts
    np.save(os.path.join(SAVE_DIR, "y_true.npy"), y_te)
    np.save(os.path.join(SAVE_DIR, "y_pred.npy"), y_pred_final)
    np.save(os.path.join(SAVE_DIR, "y_proba.npy"), P_final)

    # =========================================================================
    # STAGE 4 — FINAL PUBLICATION REPORTING (Tables 1–4 and Figures 1–3)
    # =========================================================================
    print("\n" + "="*76)
    print("STAGE 4: GENERATING FINAL PUBLICATION TABLES & 600-DPI FIGURES")
    print("="*76)

    # Table 1 (Overall Performance + 95% Stratified CI) & Table 4 (Calibration Metrics + 95% Stratified CI)
    table1_df, table4_df = bootstrap_final_model_ci(
        y_te, P_final, n_class=n_class, n_boot=N_BOOTSTRAP, seed=RANDOM_STATE
    )

    # Table 2 (Per-Class Classification Metrics: Precision, Sensitivity, Specificity, F1, Support)
    table2_df = build_per_class_report_with_specificity(y_te, y_pred_final, class_names)

    # Table 3 (Class-Wise ROC-AUC + 95% Stratified CI)
    table3_df = bootstrap_classwise_roc_auc(
        y_te, P_final, class_names=class_names, n_boot=N_BOOTSTRAP, seed=RANDOM_STATE
    )

    print("\n--- [TABLE 1: OVERALL PERFORMANCE + 95% STRATIFIED BOOTSTRAP CI] ---")
    print(table1_df[["Metric", "Formatted_95CI"]].to_string(index=False))

    print("\n--- [TABLE 2: PER-CLASS CLASSIFICATION METRICS] ---")
    print(table2_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    print("\n--- [TABLE 3: CLASS-WISE ROC-AUC + 95% STRATIFIED BOOTSTRAP CI] ---")
    print(table3_df[["Class", "Support", "Formatted_95CI"]].to_string(index=False))

    print(f"\n--- [TABLE 4: CALIBRATION METRICS ({N_CALIB_BINS}-BIN ECE, NLL, BRIER) + 95% STRATIFIED CI] ---")
    print(table4_df[["Metric", "Formatted_95CI"]].to_string(index=False))

    fig_dir   = os.path.join(SAVE_DIR, "PUBLICATION_FIGURES")
    latex_dir = os.path.join(SAVE_DIR, "PUBLICATION_LATEX")

    plot_figure1_confusion_matrices(fig_dir, y_te, y_pred_final, class_names)
    plot_figure2_roc_curves(fig_dir, y_te, P_final, class_names)
    plot_figure3_reliability_diagram(fig_dir, y_te, P_final, n_class=n_class, n_bins=N_CALIB_BINS)
    export_publication_tables(SAVE_DIR, latex_dir, table1_df, table2_df, table3_df, table4_df)

    print("\n" + "="*76)
    print(f"FINAL PUBLICATION DELIVERABLES COMPLETED -> {SAVE_DIR}")
    print("  * Artifacts : learned_parameters.json, y_true.npy, y_pred.npy, y_proba.npy")
    print(f"  * Tables    : Table1..Table4 (.csv & .tex in {latex_dir})")
    print(f"  * Figures   : Fig1..Fig3 (600-DPI .png, .pdf, .svg in {fig_dir})")
    print("="*76)

if __name__ == "__main__":
    main()